# 非评分实验：Decision Trees


本 Notebook 将可视化 Decision Tree 如何根据 Information Gain 进行分裂。

我们将重新使用视频课程中的动物数据集。


课程介绍过，Decision Tree 通过某次分裂带来的 **Information Gain（信息增益）** 判断是否分裂节点：

$$\text{Information Gain}=H(p_1^{\text{node}})-\left(w^{\text{left}}H(p_1^{\text{left}})+w^{\text{right}}H(p_1^{\text{right}})\right).$$

其中 $H$ 是 Entropy（熵）：

$$H(p_1)=-p_1\log_2(p_1)-(1-p_1)\log_2(1-p_1).$$

这里的对数以 2 为底，并约定 $0\log_2 0=0$。运行下面的代码，观察 $H(p)$ 随 $p$ 的变化。

$p=0.5$ 时，Entropy 最大，事件是否发生最不确定；$p=0$ 或 $p=1$ 时，Entropy 最小，结果完全可预测。因此，Entropy 衡量事件的不确定性。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import *

In [ ]:
%matplotlib widget
_ = plot_entropy()


| 动物 | 耳朵形状 | 脸形 | 胡须 | 是否为猫 |
| :---: | :---: | :---: | :---: | :---: |
| <img src="images/0.png" alt="动物 0" width="50"> | 尖耳 | 圆脸 | 有 | 1 |
| <img src="images/1.png" alt="动物 1" width="50"> | 垂耳 | 非圆脸 | 有 | 1 |
| <img src="images/2.png" alt="动物 2" width="50"> | 垂耳 | 圆脸 | 无 | 0 |
| <img src="images/3.png" alt="动物 3" width="50"> | 尖耳 | 非圆脸 | 有 | 0 |
| <img src="images/4.png" alt="动物 4" width="50"> | 尖耳 | 圆脸 | 有 | 1 |
| <img src="images/5.png" alt="动物 5" width="50"> | 尖耳 | 圆脸 | 无 | 1 |
| <img src="images/6.png" alt="动物 6" width="50"> | 垂耳 | 非圆脸 | 无 | 0 |
| <img src="images/7.png" alt="动物 7" width="50"> | 尖耳 | 圆脸 | 无 | 1 |
| <img src="images/8.png" alt="动物 8" width="50"> | 垂耳 | 圆脸 | 无 | 0 |
| <img src="images/9.png" alt="动物 9" width="50"> | 垂耳 | 圆脸 | 无 | 0 |

将这些二元 Categorical Features 编码为 0/1：

- Ear Shape：尖耳为 1，垂耳为 0。
- Face Shape：圆脸为 1，非圆脸为 0。
- Whiskers：有胡须为 1，无胡须为 0。

因此，`X_train` 的每个样本包含上述三个特征；`y_train` 表示动物是否为猫，猫为 1，否则为 0。


In [ ]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [ ]:
#For instance, the first example
X_train[0]

这表示第一个样本具有尖耳、圆脸和胡须。


在每个节点上，比较分裂前的 Entropy 与两个子节点的 Weighted Entropy，计算各特征的 Information Gain，然后选择 Information Gain 最高的特征进行分裂。


Root Node 包含数据集中的所有动物。$p_1^{\text{node}}$ 是其中 Positive Class（猫）的比例：

$$p_1^{\text{node}}=\frac{5}{10}=0.5.$$

下面编写计算 Entropy 的函数。


In [ ]:
def entropy(p):
    if p == 0 or p == 1:
        return 0
    else:
        return -p * np.log2(p) - (1- p)*np.log2(1 - p)
    
print(entropy(0.5))

为了说明分裂过程，下面分别计算按各特征分裂时的 Information Gain。先编写几个辅助函数。


In [ ]:
def split_indices(X, index_feature):
    """Given a dataset and a index feature, return two lists for the two split nodes, the left node has the animals that have 
    that feature = 1 and the right node those that have the feature = 0 
    index feature = 0 => ear shape
    index feature = 1 => face shape
    index feature = 2 => whiskers
    """
    left_indices = []
    right_indices = []
    for i,x in enumerate(X):
        if x[index_feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices

如果选择 Ear Shape 进行分裂，根据上表，左子节点应包含以下索引：

$$0\quad3\quad4\quad5\quad7$$

其余索引归入右子节点。


In [ ]:
split_indices(X_train, 0)

还需要一个函数，计算分裂后子节点的 Weighted Entropy。为此需要：

- $w^{\text{left}}$ 和 $w^{\text{right}}$：各子节点的样本数占父节点样本数的比例。
- $p^{\text{left}}$ 和 $p^{\text{right}}$：各子节点内部猫的比例。

注意这两组比例的分母不同。例如，按索引为 0 的 Ear Shape 特征分裂 Root Node，左子节点包含动物 0、3、4、5、7，因此：

$$w^{\text{left}}=\frac{5}{10}=0.5,\qquad p^{\text{left}}=\frac{4}{5},$$

$$w^{\text{right}}=\frac{5}{10}=0.5,\qquad p^{\text{right}}=\frac{1}{5}.$$


In [ ]:
def weighted_entropy(X,y,left_indices,right_indices):
    """
    This function takes the splitted dataset, the indices we chose to split and returns the weighted entropy.
    """
    w_left = len(left_indices)/len(X)
    w_right = len(right_indices)/len(X)
    p_left = sum(y[left_indices])/len(left_indices)
    p_right = sum(y[right_indices])/len(right_indices)
    
    weighted_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)
    return weighted_entropy

In [ ]:
left_indices, right_indices = split_indices(X_train, 0)
weighted_entropy(X_train, y_train, left_indices, right_indices)

两个子节点的 Weighted Entropy 约为 0.72。用分裂前节点（此处为 Root Node）的 Entropy 减去它，即得到 **Information Gain**。


In [ ]:
def information_gain(X, y, left_indices, right_indices):
    """
    Here, X has the elements in the node and y is theirs respectives classes
    """
    p_node = sum(y)/len(y)
    h_node = entropy(p_node)
    w_entropy = weighted_entropy(X,y,left_indices,right_indices)
    return h_node - w_entropy

In [ ]:
information_gain(X_train, y_train, left_indices, right_indices)

下面分别计算按每个特征分裂 Root Node 时的 Information Gain：


In [ ]:
for i, feature_name in enumerate(['Ear Shape', 'Face Shape', 'Whiskers']):
    left_indices, right_indices = split_indices(X_train, i)
    i_gain = information_gain(X_train, y_train, left_indices, right_indices)
    print(f"Feature: {feature_name}, information gain if we split the root node using this feature: {i_gain:.2f}")
    

最佳分裂特征确实是 Ear Shape。运行下面的代码，查看分裂结果；无需理解这段可视化代码的具体实现。


In [ ]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=1, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

这个过程是 **Recursive（递归）** 的，即对各节点重复上述计算，直到满足停止条件，例如：

- 继续分裂将超过允许的 Tree Depth。
- 节点中只剩一个类别。
- 分裂的 Information Gain 低于阈值。


最终的 Decision Tree 如下所示：


In [ ]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=2, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

恭喜完成本 Notebook！
